<a href="https://colab.research.google.com/github/anjorisarabhai/harvard_cs50_online/blob/main/ASR1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Preprocessing**

In [ ]:
# Step 1: Install dependencies
!pip install datasets transformers torchaudio jiwer requests tqdm --quiet

# Step 2: Imports and environment setup
import os
os.environ["HF_AUDIO_DECODER"] = "ffmpeg"  # Force datasets to use ffmpeg for audio decoding

import pandas as pd
import requests
from datasets import Dataset, Audio
from transformers import WhisperProcessor
from tqdm.auto import tqdm

# Step 3: Load dataset directly from Google Sheets CSV export
sheet_id = "1bujiO2NgtHlgqPlNvYAQf5_7ZcXARlIfNX5HNb9f8cI"
gid = "1786138861"
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df = pd.read_csv(csv_url)
print(f"Loaded {len(df)} rows from Google Sheet")

# Step 4: Fetch and concatenate segmented transcriptions from JSON
def fetch_transcript(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if isinstance(data, list):
            full_text = " ".join(segment.get("text", "").strip() for segment in data if "text" in segment)
            return full_text.strip()
        elif isinstance(data, dict):
            for key in ['transcription', 'text', 'sentence', 'label']:
                if key in data and isinstance(data[key], str) and data[key].strip():
                    return data[key].strip()
            return ""
        else:
            return ""
    except Exception as e:
        print(f"Warning: failed to fetch or parse transcription from {url}. Error: {e}")
        return ""

tqdm.pandas(desc="Fetching transcriptions")
df['transcription'] = df['transcription_url_gcp'].progress_apply(fetch_transcript)

# Step 5: Filter rows with valid transcriptions only
df_valid = df[df['transcription'].str.strip() != ""].reset_index(drop=True)
print(f"Number of valid transcriptions: {len(df_valid)}")
if len(df_valid) == 0:
    raise RuntimeError("No valid transcriptions found.")

# Step 6: Create a Hugging Face Dataset, stream audio from URLs
data = Dataset.from_pandas(df_valid[['rec_url_gcp', 'transcription']].rename(
    columns={'rec_url_gcp': 'audio', 'transcription': 'text'}
))

# Cast audio column with streaming to decode on-demand using ffmpeg
data = data.cast_column("audio", Audio(sampling_rate=16000, decode=True))
print(f"Dataset size: {len(data)}")

# Step 7: Load Whisper processor
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

# Step 8: Define preprocessing function
def preprocess(batch):
    audio = batch["audio"]
    inputs = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=batch["text"],
        return_tensors="pt"
    )
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = inputs.labels[0]
    return batch

# Map preprocess function (batched=False for individual decoding)
data = data.map(preprocess, remove_columns=data.column_names, batched=False)
print(f"Dataset size after preprocessing: {len(data)}")

# Step 9: Filter out any failures (if any)
data = data.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)
print(f"Dataset size after filtering invalid entries: {len(data)}")

if len(data) == 0:
    raise RuntimeError("No valid samples after preprocessing.")

# Step 10: View example processed item
print("Example processed record:")
print(data[0])

# Step 11: Save the preprocessed dataset
data.save_to_disk("preprocessed_whisper_hi")
print("Preprocessed dataset saved to 'preprocessed_whisper_hi'")

Loaded 104 rows from Google Sheet


Fetching transcriptions:   0%|          | 0/104 [00:00<?, ?it/s]

Number of valid transcriptions: 104
Dataset size: 104


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/104 [00:00<?, ? examples/s]

...

Dataset size after preprocessing: 104


Filter:   0%|          | 0/104 [00:00<?, ? examples/s]

Dataset size after filtering invalid entries: 104
Example processed record:
{'input_features': [[-0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.5645561218261719, -0.34945249557495117, -0.39926671981811523, -0.3851982355117798, -0.44325029850006104, -0.3238581418991089, -0.7618286609649658, -0.7840303182601929, -0.457120418548584, -0.3838289976119995, -0.2632852792739868, -0.37055933475494385, -0.5761399269104004, -0.771905779838562, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.7840303182601929, -0.6665658950805664, -0.44675254821777344, -0.4611915349960327, -0.43570518493652344, -0.1543104648590088, -0.12251877784729004, -0.21922659873962402, -0.26175975799560547, -0.44439947605133057, -0.4990893602371216, -0.41054093837738037, -0.4708418846130371, -0.499122381

Saving the dataset (0/1 shards):   0%|          | 0/104 [00:00<?, ? examples/s]

Preprocessed dataset saved to 'preprocessed_whisper_hi'


In [ ]:
!pip install evaluate --quiet

In [ ]:
!pip install --upgrade transformers --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save preprocessed dataset to Google Drive folder
data.save_to_disk("/content/drive/MyDrive/preprocessed_whisper_hi")

Mounted at /content/drive


Saving the dataset (0/1 shards):   0%|          | 0/104 [00:00<?, ? examples/s]

In [ ]:
import shutil

# Compress the preprocessed dataset folder
shutil.make_archive("preprocessed_whisper_hi", 'zip', "preprocessed_whisper_hi")

# Then download it locally
from google.colab import files
files.download("preprocessed_whisper_hi.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from datasets import load_from_disk
data = load_from_disk("/content/drive/MyDrive/preprocessed_whisper_hi")
print(f"Reloaded preprocessed dataset with {len(data)} samples")

Reloaded preprocessed dataset with 104 samples


In [ ]:
!pip install datasets transformers evaluate jiwer --quiet

### **Fine Tuning**

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Install required packages (if not already)
!pip install datasets transformers evaluate jiwer --quiet

# Step 3: Imports
import torch
from datasets import load_from_disk
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
import pandas as pd

# Step 4: Load preprocessed dataset from Drive
data = load_from_disk("/content/drive/MyDrive/preprocessed_whisper_hi")
print(f"Loaded preprocessed dataset with {len(data)} samples")

# Step 5: Load Whisper processor and model
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Step 6: Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper_finetuned_hi",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=1e-5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
)

# Step 7: Define data collator with label truncation and padding
max_label_length = 448  # Whisper-small allowed max target length

def data_collator(features):
    input_features = torch.stack([
        torch.tensor(f["input_features"]) if not isinstance(f["input_features"], torch.Tensor) else f["input_features"]
        for f in features
    ])

    labels = []
    for f in features:
        labels_tensor = f["labels"] if isinstance(f["labels"], torch.Tensor) else torch.tensor(f["labels"])
        if labels_tensor.size(0) > max_label_length:
            labels_tensor = labels_tensor[:max_label_length]
        labels.append(labels_tensor)

    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_features": input_features,
        "labels": labels_padded,
    }

# Step 8: Create Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=data,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator
)

# Step 9: Fine-tune the model
trainer.train()

# Step 10: Save the fine-tuned model and processor to Google Drive
trainer.save_model("/content/drive/MyDrive/whisper_finetuned_hi")
processor.save_pretrained("/content/drive/MyDrive/whisper_finetuned_hi")

print("Fine-tuning complete. Model and processor saved to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded preprocessed dataset with 104 samples


/tmp/ipython-input-572033425.py:61: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3867: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Fine-tuning complete. Model and processor saved to Google Drive.


In [ ]:
!pip install --force-reinstall datasets==2.15.0 --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.0.dev0 requires huggingface-hub==1.0.0.rc1, but you have huggingface-hub 0.35.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0

In [ ]:
# ==============================================================================
# Step 1: Imports and Configuration (Run AFTER runtime restart)
# ==============================================================================
import torch
import jiwer
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    pipeline
)
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# --- Configuration ---
FLEURS_LANG_CODE = "hi_in"  # Hindi (India) config for FLEURS
TEST_SPLIT = "test"
BASE_MODEL_ID = "openai/whisper-small"
FINETUNED_MODEL_PATH = "/content/drive/MyDrive/whisper_finetuned_hi"

# Set device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ==============================================================================
# Step 2: Load and Preprocess the FLEURS Dataset
# ==============================================================================

print(f"Loading FLEURS dataset ({FLEURS_LANG_CODE}, split: {TEST_SPLIT})...")
# This should now work due to the datasets==2.15.0 installation
fleurs_dataset = load_dataset("google/fleurs", FLEURS_LANG_CODE, split=TEST_SPLIT)

# Whisper models require audio to be 16kHz
fleurs_dataset = fleurs_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)
print(f"Loaded {len(fleurs_dataset)} samples from FLEURS.")

# --- Define the Preprocessing Function for the Evaluation Pipeline ---
def prepare_data_for_pipeline(batch):
    # This prepares the audio array and keeps the transcription
    audio = batch["audio"]
    batch["audio_array"] = audio["array"]
    batch["sampling_rate"] = audio["sampling_rate"]
    # The reference text must be in a 'text' column for the pipeline
    batch["text"] = batch["raw_transcription"]
    return batch

# Rename the reference column before casting to 'text' for the pipeline
fleurs_dataset = fleurs_dataset.rename_column("transcription", "raw_transcription")
# Map the function to get the raw audio array
fleurs_dataset = fleurs_dataset.map(
    prepare_data_for_pipeline,
    remove_columns=fleurs_dataset.column_names,
    num_proc=1 # Set to >1 for faster loading on multi-core systems
)

# ==============================================================================
# Step 3: Define Evaluation Function (Inference & WER Calculation)
# ==============================================================================

def evaluate_model_wer(model_id_or_path, dataset, lang_code, device):
    """
    Performs inference and calculates WER for a given model on a dataset.
    """
    print(f"\nEvaluating model: {model_id_or_path}...")

    # Load processor and model
    try:
        processor = WhisperProcessor.from_pretrained(model_id_or_path)
        model = WhisperForConditionalGeneration.from_pretrained(model_id_or_path)

        # Configure model generation for Hindi transcription
        model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
            language=lang_code.split('_')[0], # Use 'hi' for language prompt
            task="transcribe"
        )
        model.config.suppress_tokens = []
    except Exception as e:
        print(f"Error loading model/processor from {model_id_or_path}. Error: {e}")
        return None

    # Use the Hugging Face ASR pipeline for efficient batch inference
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model.to(device), # Move model to device
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
        batch_size=16 # Adjust based on your GPU memory
    )

    # Extract audio arrays for the pipeline
    audio_inputs = [
        {"array": item["audio_array"], "sampling_rate": 16000}
        for item in dataset
    ]

    # Transcribe
    transcriptions = pipe(
        audio_inputs,
        chunk_length_s=30,
        stride_length_s=[4, 2],
        generate_kwargs={"language": lang_code.split('_')[0], "task": "transcribe"},
        return_timestamps=False,
    )

    # Extract predicted text
    predictions = [item['text'] for item in transcriptions]

    # Extract reference text
    references = dataset["text"]

    # Calculate WER
    wer = jiwer.wer(references, predictions)

    # WER is typically reported as a percentage
    return wer * 100

# ==============================================================================
# Step 4: Run Evaluation for Both Models
# ==============================================================================

results = {}

# 1. Evaluate Pre-trained Baseline
wer_baseline = evaluate_model_wer(
    BASE_MODEL_ID, fleurs_dataset, FLEURS_LANG_CODE, device
)
results["Whisper-small (Pre-trained)"] = wer_baseline

# 2. Evaluate Fine-Tuned Model
wer_finetuned = evaluate_model_wer(
    FINETUNED_MODEL_PATH, fleurs_dataset, FLEURS_LANG_CODE, device
)
results["Whisper-small (Fine-tuned)"] = wer_finetuned

# ==============================================================================
# Step 5: Report Results in Structured Table
# ==============================================================================

print("\n" + "="*50)
print("             WHISPER MODEL WER RESULTS (FLEURS - Hindi)")
print("="*50)

df_results = pd.DataFrame({
    "Model": list(results.keys()),
    "WER (%)": [f"{v:.2f}" if v is not None else "N/A" for v in results.values()]
})

# Display the table
display(Markdown(df_results.to_markdown(index=False)))

if wer_finetuned is not None and wer_baseline is not None:
    improvement = wer_baseline - wer_finetuned
    print(f"\n✅ Fine-tuning resulted in an absolute WER reduction of: {improvement:.2f} percentage points.")
elif wer_baseline is not None:
    print("\n⚠️ Fine-tuned model WER is N/A. Check if model path is correct and Drive is mounted.")
else:
    print("\n⚠️ Could not calculate any WER. Check library installation and model paths.")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda:0
Loading FLEURS dataset (hi_in, split: test)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loaded 418 samples from FLEURS.


ValueError: New column name raw_transcription already in the dataset. Please choose a column name which is not already in the dataset. Current columns in the dataset: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']

In [ ]:
# 1. Install a known stable version of transformers and force the hub version
!pip install --upgrade transformers==4.35.2 huggingface-hub==0.20.0 --quiet

# 2. Re-force the specific datasets version needed to load FLEURS
# (This is still necessary to bypass the original dataset loading error)
!pip install --force-reinstall datasets==2.15.0 --quiet

# 3. Install jiwer again just in case it was downgraded/removed
!pip install jiwer --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.46.0 requires huggingface-hub<1.0,>=0.33.5, but you have huggingface-hub 0.20.0 which is incompatible.
sentence-transformers 5.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
peft 0.17.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.20.0 which is incompatible.
diffusers 0.35.1 requires huggingface-hub>=0.34.0, but you have huggingface-hub 0.20.0 which is incompatible.
accelerate 1.10.1 requires huggingface_hub>=0.21.0, but yo

In [ ]:
!pip uninstall numba -y

Found existing installation: numba 0.60.0
Uninstalling numba-0.60.0:
  Successfully uninstalled numba-0.60.0


In [ ]:
# 1. Install librosa, which will bring Numba back, but hopefully a compatible version.
!pip install librosa==0.10.1 --quiet

# 2. Re-force the specific datasets and transformers versions to maintain stability.
!pip install --force-reinstall datasets==2.15.0 transformers==4.35.2 torchaudio jiwer --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires numba<0.62.0a0,>=0.59.1, but you have numba 0.62.0 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cuml-cu12 25.6.0 requires numba<0.62.0a0,>=0.59.1, but you have numba 0.62.0 which is incompatible.
distributed-ucxx-cu12 0.44.0 requires numba<0.62.0a0,>=0.59.1, but you have numba 0.62.0 which is incompatible.
dask-cuda 25.6.0 requires numba<0.62.0a0,>=0

## **Evaluation**

In [ ]:
# ==============================================================================
# Step 1: Imports and Configuration (Run AFTER the final restart)
# ==============================================================================
import torch
import jiwer
from datasets import load_dataset, Audio, Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# --- Configuration ---
FLEURS_LANG_CODE = "hi_in"  # Hindi (India) config for FLEURS
TEST_SPLIT = "test"
BASE_MODEL_ID = "openai/whisper-small"
FINETUNED_MODEL_PATH = "/content/drive/MyDrive/whisper_finetuned_hi"

# Set device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ==============================================================================
# Step 2: Load and Prepare the FLEURS Dataset
# ==============================================================================

print(f"Loading FLEURS dataset ({FLEURS_LANG_CODE}, split: {TEST_SPLIT})...")

# Reload dataset is required after kernel restart
fleurs_dataset = load_dataset("google/fleurs", FLEURS_LANG_CODE, split=TEST_SPLIT)

# Whisper models require audio to be 16kHz
fleurs_dataset = fleurs_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)
print(f"Loaded {len(fleurs_dataset)} samples from FLEURS.")

# We drop unnecessary columns.
fleurs_dataset = fleurs_dataset.remove_columns([
    'id', 'num_samples', 'path', 'raw_transcription',
    'gender', 'lang_id', 'language', 'lang_group_id'
])


# ==============================================================================
# Step 3: Define Evaluation Function (Manual Inference & WER Calculation)
# ==============================================================================

def evaluate_model_wer_manual(model_id_or_path, dataset, lang_code, device):
    """
    Performs inference manually over the dataset to avoid pipeline/map dependencies.
    """
    print(f"\nEvaluating model: {model_id_or_path}...")

    # Load processor and model
    try:
        processor = WhisperProcessor.from_pretrained(model_id_or_path)
        model = WhisperForConditionalGeneration.from_pretrained(model_id_or_path).to(device)
        model.eval()
    except Exception as e:
        print(f"Error loading model/processor from {model_id_or_path}. Error: {e}")
        return None

    # Set generation config for Hindi transcription
    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=lang_code.split('_')[0],  # Use 'hi'
        task="transcribe"
    )

    predictions = []
    references = []

    # Manually iterate over each sample (no datasets.map, no pipeline)
    for sample in tqdm(dataset, desc="Transcribing"):
        audio_array = sample["audio"]["array"]
        reference_text = sample["transcription"]

        # 1. Prepare input features
        # Note: This step uses the processor, which should now work with the fixed dependencies
        input_features = processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(device)

        # 2. Generate transcription
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids
            )

        # 3. Decode prediction and store
        prediction_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        predictions.append(prediction_text)
        references.append(reference_text)

    # Calculate WER
    wer = jiwer.wer(references, predictions)

    return wer * 100

# ==============================================================================
# Step 4: Run Evaluation for Both Models
# ==============================================================================

results = {}

# 1. Evaluate Pre-trained Baseline
wer_baseline = evaluate_model_wer_manual(
    BASE_MODEL_ID, fleurs_dataset, FLEURS_LANG_CODE, device
)
results["Whisper-small (Pre-trained)"] = wer_baseline

# 2. Evaluate Fine-Tuned Model
wer_finetuned = evaluate_model_wer_manual(
    FINETUNED_MODEL_PATH, fleurs_dataset, FLEURS_LANG_CODE, device
)
results["Whisper-small (Fine-tuned)"] = wer_finetuned

# ==============================================================================
# Step 5: Report Results in Structured Table
# ==============================================================================

print("\n" + "="*50)
print("             WHISPER MODEL WER RESULTS (FLEURS - Hindi)")
print("="*50)

df_results = pd.DataFrame({
    "Model": list(results.keys()),
    "WER (%)": [f"{v:.2f}" if v is not None else "N/A" for v in results.values()]
})

# Display the table
display(Markdown(df_results.to_markdown(index=False)))

if wer_finetuned is not None and wer_baseline is not None:
    improvement = wer_baseline - wer_finetuned
    print(f"\n✅ Fine-tuning resulted in an absolute WER reduction of: {improvement:.2f} percentage points.")
elif wer_baseline is not None:
    print("\n⚠️ Fine-tuned model WER is N/A. Check if model path is correct and Drive is mounted.")
else:
    print("\n⚠️ Could not calculate any WER.")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda:0
Loading FLEURS dataset (hi_in, split: test)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded 418 samples from FLEURS.

Evaluating model: openai/whisper-small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Transcribing:   0%|          | 0/418 [00:00<?, ?it/s]


             WHISPER MODEL WER RESULTS (FLEURS - Hindi)


| Model                       |   WER (%) |
|:----------------------------|----------:|
| Whisper-small (Pre-trained) |     84.16 |
| Whisper-small (Fine-tuned)  |    273.8  |


✅ Fine-tuning resulted in an absolute WER reduction of: -189.64 percentage points.


In [ ]:
from datasets import load_dataset, Audio

FLEURS_LANG_CODE = "hi_in"
TEST_SPLIT = "test"

print(f"Loading FLEURS dataset ({FLEURS_LANG_CODE}, split: {TEST_SPLIT})...")

fleurs_dataset = load_dataset("google/fleurs", FLEURS_LANG_CODE, split=TEST_SPLIT)

# Cast to 16kHz to match Whisper requirement
fleurs_dataset = fleurs_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

# You can drop columns not required during evaluation
fleurs_dataset = fleurs_dataset.remove_columns([
    'id', 'num_samples', 'path', 'raw_transcription',
    'gender', 'lang_id', 'language', 'lang_group_id'
])

print(f"Loaded {len(fleurs_dataset)} samples from FLEURS.")

Loading FLEURS dataset (hi_in, split: test)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded 418 samples from FLEURS.


In [ ]:
import torch
from tqdm.auto import tqdm
import jiwer
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from IPython.display import display

# Updated normalization transforms for WER calculation (no RemoveShortWords)
transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.RemoveEmptyStrings(),
])

def evaluate_model_wer_manual_normalized(model_id_or_path, dataset, lang_code, device, num_samples=None):
    """
    Manually runs model inference on dataset, returns normalized WER and example pairs.
    """
    processor = WhisperProcessor.from_pretrained(model_id_or_path)
    model = WhisperForConditionalGeneration.from_pretrained(model_id_or_path).to(device)
    model.eval()

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=lang_code.split('_')[0],
        task="transcribe"
    )

    predictions = []
    references = []

    # Limit samples if specified, to speed up debugging
    if num_samples is not None:
        dataset = (x for i, x in enumerate(dataset) if i < num_samples)

    for sample in tqdm(dataset, desc="Transcribing"):
        audio_array = sample["audio"]["array"]
        reference_text = sample["transcription"]

        input_features = processor(
            audio_array, sampling_rate=16000, return_tensors="pt"
        ).input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

        prediction_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        predictions.append(prediction_text)
        references.append(reference_text)

    # Normalize texts before WER calculation
    normalized_refs = [transformation(ref) for ref in references]
    normalized_preds = [transformation(pred) for pred in predictions]

    wer = jiwer.wer(normalized_refs, normalized_preds)

    # Print some example pairs for manual inspection
    print("\nExample transcriptions (reference | prediction):\n")
    for i in range(min(5, len(normalized_refs))):
        print(f"Ref: {normalized_refs[i]}")
        print(f"Pred: {normalized_preds[i]}")
        print("---")

    return wer * 100  # percent

# Usage example assuming fleurs_dataset loaded and device set:

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model_id = "openai/whisper-small"
finetuned_model_path = "/content/drive/MyDrive/whisper_finetuned_hi"
fleurs_lang_code = "hi_in"

print("Evaluating baseline model:")
wer_baseline = evaluate_model_wer_manual_normalized(base_model_id, fleurs_dataset, fleurs_lang_code, device, num_samples=100)

print("\nEvaluating fine-tuned model:")
wer_finetuned = evaluate_model_wer_manual_normalized(finetuned_model_path, fleurs_dataset, fleurs_lang_code, device, num_samples=100)

# Display final results
df_results = pd.DataFrame({
    "Model": ["Whisper-small Baseline", "Whisper-small Fine-tuned"],
    "WER (%)": [f"{wer_baseline:.2f}", f"{wer_finetuned:.2f}"]
})
display(df_results)

Evaluating baseline model:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Transcribing: 0it [00:00, ?it/s]


Example transcriptions (reference | prediction):

Ref: कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है
Pred: अग्वो में आज्द्टर केंद्रख होता है जिसका मतला भी आजा की उन्मे थोडे या बिना किसी जटके से तुटनें की प्रवत्ती होती है
---
Ref: ग्रीनलैंड को बहुत कम जगह बसाया गया था नॉर्स सगास में वे कहते हैं कि एरिक रेड हत्या के लिए आइसलैंड से निर्वासित किया गया था और आगे पश्चिम की यात्रा करते समय ग्रीनलैंड मिला जिसे ग्रीनलैंड नाम दिया गया
Pred: गरीलेंड को बहुत कम जग़़ बसाया गया ता नोर्ष शगास में भे कहते हैं कि एरेक रेड रेद हत्या कि लिए आइस लेंड से निरवासित की आप या आप और आगे पश्च्ट्म की याट्र करते समय गरीलेंड मिला जिसे गरीलेंड नाम दिया गया
---
Ref: ऐसी कोई वैश्विक परिभाषा नहीं है जिसके लिए निर्मित सामान एंटीक होते हैं कुछ कर एजेंसियां 100 साल से पुराने सामान को एंटीक के तौर पर परिभाषित करती हैं
Pred: अएजेई कोई वैस्पिक परिवाशा नहीं है जिसके लिए निरमिद समान अंटिक होते है पुचकर अजंश्या स्वाव साल से पुरानि समान को अंटिक के तोर पर पर

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import torch
from transformers import WhisperConfig, WhisperForConditionalGeneration, WhisperProcessor

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load processor from local path
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Load model config from local path
config = WhisperConfig.from_json_file(f"{finetuned_path}/config.json")

# Initialize the model from the config
finetuned_model = WhisperForConditionalGeneration(config)

# Load model weights from local file
state_dict = torch.load(f"{finetuned_path}/pytorch_model.bin")
finetuned_model.load_state_dict(state_dict)

finetuned_model.to("cuda" if torch.cuda.is_available() else "cpu")
finetuned_model.eval()

import torch
from tqdm.auto import tqdm
import jiwer

# Updated normalization transforms for WER calculation
transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.RemoveEmptyStrings(),
])

def evaluate_model_wer_manual_normalized(model, processor, dataset, lang_code, device, num_samples=None):
    """
    Manually runs model inference on dataset using given model and processor,
    returns normalized WER and prints example transcriptions.
    """
    model.eval()

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=lang_code.split('_')[0],
        task="transcribe"
    )

    predictions = []
    references = []

    if num_samples is not None:
        dataset = (x for i, x in enumerate(dataset) if i < num_samples)

    for sample in tqdm(dataset, desc="Transcribing"):
        audio_array = sample["audio"]["array"]
        reference_text = sample["transcription"]

        input_features = processor(
            audio_array, sampling_rate=16000, return_tensors="pt"
        ).input_features.to(device)

        with torch.no_grad():
            generated_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

        prediction_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        predictions.append(prediction_text)
        references.append(reference_text)

    normalized_refs = [transformation(ref) for ref in references]
    normalized_preds = [transformation(pred) for pred in predictions]

    wer = jiwer.wer(normalized_refs, normalized_preds)

    print("\nExample transcriptions (reference | prediction):\n")
    for i in range(min(5, len(normalized_refs))):
        print(f"Ref: {normalized_refs[i]}")
        print(f"Pred: {normalized_preds[i]}")
        print("---")

    return wer * 100  # return WER as percentage

# Assuming you already loaded fleurs_dataset and device

print("\nEvaluating fine-tuned model:")

wer_finetuned = evaluate_model_wer_manual_normalized(
    finetuned_model,
    processor,
    fleurs_dataset,
    fleurs_lang_code,
    device,
    num_samples=100
)

print(f"Fine-tuned model WER: {wer_finetuned:.2f}%")

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import os
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load processor normally with local_files_only
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# The trick: specify full directory and force local files only to avoid HF Hub validation
finetuned_model = WhisperForConditionalGeneration.from_pretrained(
    finetuned_path,
    local_files_only=True,
    from_tf=False
)

finetuned_model.to("cuda" if torch.cuda.is_available() else "cpu")
finetuned_model.eval()

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import os
import torch
from transformers import WhisperProcessor, WhisperConfig, WhisperForConditionalGeneration

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load processor locally
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Load config from JSON file explicitly
config_path = os.path.join(finetuned_path, "config.json")
config = WhisperConfig.from_json_file(config_path)

# Initialize model from config
finetuned_model = WhisperForConditionalGeneration(config)

# Load model weights manually (adjust filename if different)
weights_path = os.path.join(finetuned_path, "pytorch_model.bin")
state_dict = torch.load(weights_path, map_location='cpu')

# Load weights into model
finetuned_model.load_state_dict(state_dict)

finetuned_model.to("cuda" if torch.cuda.is_available() else "cpu")
finetuned_model.eval()

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, WhisperConfig
import os

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Step 1: Load processor with local_files_only=True to avoid hub calls
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Step 2: Load config manually from JSON file
config = WhisperConfig.from_json_file(os.path.join(finetuned_path, "config.json"))

# Step 3: Initialize a fresh model with config
model = WhisperForConditionalGeneration(config)

# Step 4: Load state dict weights manually with torch.load
state_dict = torch.load(os.path.join(finetuned_path, "pytorch_model.bin"), map_location="cpu")
model.load_state_dict(state_dict)

# Step 5: Move model to device and set eval mode
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import torch
import json
from transformers import WhisperProcessor, WhisperConfig, WhisperForConditionalGeneration
import os

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load processor normally using local_files_only=True (prevents Hub lookup)
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Load config.json manually
config_path = os.path.join(finetuned_path, "config.json")
with open(config_path, "r") as f:
    config_dict = json.load(f)
config = WhisperConfig.from_dict(config_dict)

# Instantiate model from config
model = WhisperForConditionalGeneration(config)

# Load PyTorch weights manually (ensure filename is correct)
weights_path = os.path.join(finetuned_path, "pytorch_model.bin")
state_dict = torch.load(weights_path, map_location="cpu")
model.load_state_dict(state_dict)

# Move to device and eval mode
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()


OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import torch
import json
from transformers import WhisperProcessor, WhisperConfig, WhisperForConditionalGeneration
import os

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load WhisperProcessor with local_files_only=True to avoid Hub calls
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Read the config json file manually
with open(os.path.join(finetuned_path, "config.json"), "r") as f:
    config_dict = json.load(f)

config = WhisperConfig.from_dict(config_dict)

# Instantiate the model with this config (no from_pretrained)
model = WhisperForConditionalGeneration(config)

# Load the PyTorch state dict manually from local file
state_dict = torch.load(os.path.join(finetuned_path, "pytorch_model.bin"), map_location="cpu")
model.load_state_dict(state_dict)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import torch
import json
import os
from transformers import WhisperProcessor, WhisperConfig, WhisperForConditionalGeneration

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load processor (this still works with local_files_only)
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Load config manually via json
with open(os.path.join(finetuned_path, "config.json"), "r") as f:
    config_dict = json.load(f)
config = WhisperConfig.from_dict(config_dict)

# Initialize model with config (no from_pretrained)
model = WhisperForConditionalGeneration(config)

# Load weights manually with torch.load
weights_path = os.path.join(finetuned_path, "pytorch_model.bin")
state_dict = torch.load(weights_path, map_location="cpu")
model.load_state_dict(state_dict)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Now model and processor are ready for evaluation


OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import os
import json
import torch
from transformers import WhisperProcessor, WhisperConfig, WhisperForConditionalGeneration

# Path to your fine-tuned model directory in Google Drive
finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load processor normally
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Manually read config.json file to a dictionary
with open(os.path.join(finetuned_path, "config.json"), "r") as f:
    config_dict = json.load(f)

# Create config object from dictionary
config = WhisperConfig.from_dict(config_dict)

# Instantiate model with this config (DO NOT use from_pretrained here)
model = WhisperForConditionalGeneration(config)

# Load model weights manually from pytorch_model.bin
weights_path = os.path.join(finetuned_path, "pytorch_model.bin")
state_dict = torch.load(weights_path, map_location="cpu")
model.load_state_dict(state_dict)

# Move to appropriate device and set to eval mode
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
import os
import json
import torch
from transformers import WhisperProcessor, WhisperConfig, WhisperForConditionalGeneration

finetuned_path = "/content/drive/MyDrive/whisper_finetuned_hi"

# Load processor (local_files_only=True to avoid Hub access)
processor = WhisperProcessor.from_pretrained(finetuned_path, local_files_only=True)

# Manually read config.json file
config_path = os.path.join(finetuned_path, "config.json")
with open(config_path, "r") as f:
    config_dict = json.load(f)

config = WhisperConfig.from_dict(config_dict)

# Instantiate model from config (NO from_pretrained)
model = WhisperForConditionalGeneration(config)

# Load weights manually
weights_path = os.path.join(finetuned_path, "pytorch_model.bin")
state_dict = torch.load(weights_path, map_location="cpu")
model.load_state_dict(state_dict)

# Move to device and eval
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

OSError: Incorrect path_or_model_id: '/content/drive/MyDrive/whisper_finetuned_hi'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
!pip install numpy==1.25.1 --force-reinstall
!pip install numba --force-reinstall

  Using cached numpy-1.25.1.tar.gz (10.4 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
  Using cached numba-0.62.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.8 kB)
  Using cached llvmlite-0.45.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached numpy-2.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
Using cached numba-0.62.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_6

In [ ]:
# Complete Question 3 solution: direct Google Sheets CSV read + disfluency detection + audio clipping
# Run in Colab

!pip install pandas requests pydub tqdm --quiet

import os
import json
import requests
import pandas as pd
from pydub import AudioSegment
from tqdm import tqdm

# --- CONFIGS ---

# Google Sheets CSV export URLs (replace if changed)
metadata_url = "https://docs.google.com/spreadsheets/d/1bujiO2NgtHlgqPlNvYAQf5_7ZcXARlIfNX5HNb9f8cI/export?format=csv&gid=1786138861"
disfluency_url = "https://docs.google.com/spreadsheets/d/1Xmm2k3Phnh89vqUolqUmAy1CeRrmB89CX_Lz8bAAe1I/export?format=csv&gid=0"

# Local directories for audio and clips
AUDIO_DIR = "/content/audio_full"
CLIP_DIR = "/content/audio_clips"
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(CLIP_DIR, exist_ok=True)

# --- LOAD DATA ---

print("Loading metadata from Google Sheets...")
metadata_df = pd.read_csv(metadata_url)
print(f"Loaded {len(metadata_df)} metadata records.")

print("Loading disfluency list from Google Sheets...")
disfluency_df = pd.read_csv(disfluency_url)
disfluencies = disfluency_df.iloc[:, 0].astype(str).str.lower().tolist()
print(f"Loaded {len(disfluencies)} disfluencies.")

# --- HELPER FUNCTIONS ---

def download_file(url, local_path):
    if os.path.exists(local_path):
        return local_path
    try:
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(local_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded {local_path}")
    except Exception as e:
        print(f"Failed to download {url}: {e}")
    return local_path

def clip_audio_segment(full_audio_path, start_sec, end_sec, clip_path):
    try:
        audio = AudioSegment.from_file(full_audio_path)
        clip = audio[start_sec*1000:end_sec*1000]
        clip.export(clip_path, format="wav")
    except Exception as e:
        print(f"Failed clipping audio {clip_path}: {e}")

def contains_disfluency(text, disfluencies):
    text_lower = text.lower()
    for d in disfluencies:
        if d in text_lower:
            return d
    return None

# --- PROCESSING LOOP ---

output_records = []

print("Processing recordings for disfluency detection and clipping...")
for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
    audio_url = row['rec_url_gcp']
    recording_id = row['recording_id']
    transcription_url = row['transcription_url_gcp']
    audio_path = os.path.join(AUDIO_DIR, f"{recording_id}.wav")

    # Download full audio if missing
    download_file(audio_url, audio_path)

    # Download and parse transcription JSON
    try:
        resp = requests.get(transcription_url)
        resp.raise_for_status()
        transcription_json = resp.json()
    except Exception as e:
        print(f"Skipping {recording_id} due to transcription download error: {e}")
        continue

    # If transcription_json is a list, use it directly as segments
    if isinstance(transcription_json, list):
        segments = transcription_json
    else:
        segments = transcription_json.get('segments', [])

    for seg in segments:
        seg_text = seg.get('text', '')
        disfluency_found = contains_disfluency(seg_text, disfluencies)
        if disfluency_found:
            start = seg.get('start')
            end = seg.get('end')
            if start is not None and end is not None and start < end:
                clip_fn = f"{recording_id}_{start:.2f}_{end:.2f}_{disfluency_found}.wav"
                clip_path = os.path.join(CLIP_DIR, clip_fn)
                clip_audio_segment(audio_path, start, end, clip_path)
                output_records.append({
                    "recording_id": recording_id,
                    "disfluency": disfluency_found,
                    "segment_text": seg_text,
                    "start_time": start,
                    "end_time": end,
                    "clip_path": clip_path,
                    "audio_url": audio_url,
                    "transcription_url": transcription_url
                })

# Save output
output_csv = "/content/disfluency_segments.csv"
pd.DataFrame(output_records).to_csv(output_csv, index=False)
print(f"\nDisfluency detection complete. Output CSV saved at {output_csv}")
print(f"Audio clips saved in {CLIP_DIR}")

Loading metadata from Google Sheets...
Loaded 104 metadata records.
Loading disfluency list from Google Sheets...
Loaded 67 disfluencies.
Processing recordings for disfluency detection and clipping...


  1%|          | 1/104 [00:01<03:23,  1.97s/it]

Downloaded /content/audio_full/825727.wav


  2%|▏         | 2/104 [00:11<10:26,  6.14s/it]

Downloaded /content/audio_full/988596.wav


...

Downloaded /content/audio_full/840793.wav


 99%|█████████▉| 103/104 [19:38<00:14, 14.77s/it]

Downloaded /content/audio_full/840781.wav


100%|██████████| 104/104 [19:56<00:00, 11.50s/it]


Disfluency detection complete. Output CSV saved at /content/disfluency_segments.csv
Audio clips saved in /content/audio_clips


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

# Change these paths as needed
csv_local_path = "/content/disfluency_segments.csv"
audio_clips_local_folder = "/content/audio_clips"
drive_folder = "/content/drive/MyDrive/"

# Copy CSV file
shutil.copy(csv_local_path, drive_folder)

# Copy audio clips folder (recursively)
shutil.copytree(audio_clips_local_folder, f"{drive_folder}/audio_clips")

'/content/drive/MyDrive//audio_clips'

In [ ]:
from google.colab import drive
import shutil
import os

# Mount Google Drive (force remount if already mounted)
drive.mount('/content/drive', force_remount=True)

# Define local and Drive paths
local_clips_folder = '/content/audio_clips'
local_csv = '/content/disfluency_segments.csv'
drive_folder = '/content/drive/MyDrive/'  # Change to your desired folder in Drive

# Copy clipped audio files folder to Drive (merge if folder exists)
shutil.copytree(local_clips_folder, os.path.join(drive_folder, 'audio_clips'), dirs_exist_ok=True)

# Copy the CSV file to Drive
shutil.copy(local_csv, os.path.join(drive_folder, 'disfluency_segments.csv'))

print(f"Copied audio clips and CSV files to {drive_folder}.")
print("Now, please go to your Google Drive folder, share the clips folder and CSV with 'Anyone with the link',")
print("and manually update the CSV 'audio_segment_url' column with shareable URLs of the clips.")

Mounted at /content/drive
Copied audio clips and CSV files to /content/drive/MyDrive/.
Now, please go to your Google Drive folder, share the clips folder and CSV with 'Anyone with the link',
and manually update the CSV 'audio_segment_url' column with shareable URLs of the clips.


In [ ]:
import pandas as pd
import re

# Load unique words CSV or use Google Sheets URL CSV export
words_url = "https://docs.google.com/spreadsheets/d/17DwCAx6Tym5Nt7eOni848np9meR-TIj7uULMtYcgQaw/export?format=csv"
df = pd.read_csv(words_url)
words = df['word'].astype(str).tolist()

# Unicode range for Devanagari block: U+0900–U+097F (basic Hindi script)
DEVANAGARI_REGEX = re.compile(r'^[\u0900-\u097F]+$')

# Common allowed punctuations or symbols in Hindi text (optional)
ALLOWED_SYMBOLS = set(['-', '_', ' '])

def is_devanagari(word):
    # True if word consists only of Devanagari chars or allowed symbols
    return all(c in ALLOWED_SYMBOLS or '\u0900' <= c <= '\u097F' for c in word)

def has_unusual_repetitions(word):
    # Check for 3 or more consecutive repeated characters as a heuristic error
    return bool(re.search(r'(.)\1{2,}', word))

def contains_latin(word):
    # Check if any Latin alphabet characters present (likely error)
    return bool(re.search(r'[A-Za-z]', word))

def check_spelling(word):
    word = word.strip()
    if not word:
        return 'incorrect spelling'
    if not is_devanagari(word):
        # Contains invalid scripts or chars
        return 'incorrect spelling'
    if contains_latin(word):
        return 'incorrect spelling'
    if has_unusual_repetitions(word):
        return 'incorrect spelling'
    # Additional heuristics or dictionary checks can go here

    # Passed all checks - mark correct
    return 'correct spelling'

# Apply spelling check to all words
df['spelling_status'] = df['word'].apply(check_spelling)

# Count correct words
num_correct = df[df['spelling_status'] == 'correct spelling'].shape[0]
print(f"Number of words marked as correct spelling: {num_correct}")

# Save final annotated CSV
output_csv = '/content/hindi_words_spelling_classification.csv'
df.to_csv(output_csv, index=False)
print(f"Saved final classification CSV at {output_csv}")

Number of words marked as correct spelling: 159365
Saved final classification CSV at /content/hindi_words_spelling_classification.csv


In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# Define local and Drive paths
local_csv_path = '/content/hindi_words_spelling_classification.csv'
drive_folder = '/content/drive/MyDrive/'  # Change to your desired folder in Drive

# Copy CSV file to Drive
shutil.copy(local_csv_path, drive_folder)

print(f"Copied CSV to Google Drive at {drive_folder}")


Mounted at /content/drive
Copied CSV to Google Drive at /content/drive/MyDrive/


In [15]:
import pandas as pd
from google.colab import drive
import shutil

# Step 1: Manually create the results DataFrame
results = {
    "Model": ["Whisper-small (Pre-trained)", "Whisper-small (Fine-tuned)"],
    "WER (%)": [84.16, 273.8]
}
df_results = pd.DataFrame(results)
print(df_results)

# Step 2: Save to CSV locally
csv_local_path = "/content/wer_results.csv"
df_results.to_csv(csv_local_path, index=False)
print(f"Saved WER results to {csv_local_path}")

# Step 3: Mount Google Drive
drive.mount('/content/drive')

# Step 4: Copy CSV to Google Drive folder (replace 'YourFolder' with your folder name)
drive_folder = "/content/drive/MyDrive/"  # Change to your Drive folder path
csv_drive_path = f"{drive_folder}/wer_results.csv"
shutil.copy(csv_local_path, csv_drive_path)
print(f"Copied WER results CSV to Google Drive at {csv_drive_path}")

                         Model  WER (%)
0  Whisper-small (Pre-trained)    84.16
1   Whisper-small (Fine-tuned)   273.80
Saved WER results to /content/wer_results.csv
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied WER results CSV to Google Drive at /content/drive/MyDrive//wer_results.csv


In [5]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset, Audio
import numpy as np

# Constants and configs
MODEL_NAME = "openai/whisper-small"
LANG_CODE = "hi"
OUTPUT_DIR = "/content/drive/MyDrive/whisper_finetuned_hi_v2"
BATCH_SIZE = 8
LEARNING_RATE = 5e-5
NUM_EPOCHS = 5
WARMUP_STEPS = 500

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Whisper processor and model
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
import pandas as pd
import requests
import json
from tqdm.auto import tqdm

# Step 1: Load your dataset CSV directly from Google Sheets export
sheet_id = "1bujiO2NgtHlgqPlNvYAQf5_7ZcXARlIfNX5HNb9f8cI"
gid = "1786138861"
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df = pd.read_csv(csv_url)
print(f"Loaded {len(df)} rows from Google Sheet")

# Step 2: Function to fetch transcript JSON and concatenate text
def fetch_transcript(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if isinstance(data, list):
            return " ".join(seg.get("text", "") for seg in data if "text" in seg).strip()
        elif isinstance(data, dict):
            for key in ['transcription', 'text', 'sentence', 'label']:
                if key in data and isinstance(data[key], str):
                    return data[key].strip()
        return ""
    except Exception as e:
        print(f"Warning: Failed to fetch transcription from {url}: {e}")
        return ""

# Step 3: Fetch all transcripts (may take time)
tqdm.pandas()
df['transcription'] = df['transcription_url_gcp'].progress_apply(fetch_transcript)

# Step 4: Filter rows with non-empty transcripts and valid audio URLs
df_valid = df[(df['transcription'].str.strip() != "") & (df['rec_url_gcp'].notna())].reset_index(drop=True)
print(f"Valid samples: {len(df_valid)}")

# Step 5: Prepare dataset dict for JSON with audio URLs and transcripts (streaming audio supported by Whisper)
dataset_json = []
for i, row in df_valid.iterrows():
    entry = {
        "audio": {"path": row['rec_url_gcp']},
        "transcription": row['transcription']
    }
    dataset_json.append(entry)

# Step 6: Save train and validation splits (80-20 split here)
split_idx = int(len(dataset_json)*0.8)
train_data = dataset_json[:split_idx]
val_data = dataset_json[split_idx:]

with open("train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open("validation.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

print(f"Saved train.json ({len(train_data)} samples) and validation.json ({len(val_data)} samples)")

Loaded 104 rows from Google Sheet


  0%|          | 0/104 [00:00<?, ?it/s]

Valid samples: 104
Saved train.json (83 samples) and validation.json (21 samples)


In [4]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "train.json",
    "validation": "validation.json"
})

In [7]:
from datasets import Audio
import torch

# Load your JSON dataset (train.json and validation.json you prepared)
dataset = load_dataset("json", data_files={
    "train": "train.json",
    "validation": "validation.json"
})

# Cast the audio to 16kHz as expected by Whisper
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Define preprocessing function to extract input_features and labels using WhisperProcessor
def preprocess(batch):
    audio = batch["audio"]["array"]
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding="longest"
    )

    # Encode labels using processor.tokenizer directly
    labels = processor.tokenizer(
        batch["transcription"],
        return_tensors="pt",
        padding="longest",
        max_length=448,
        truncation=True
    ).input_ids

    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = labels[0]

    return batch

# Preprocess datasets
dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names, batched=False)

# Define data collator to pad inputs and labels dynamically during batching
def data_collator(features):
    input_features = torch.stack([f["input_features"] for f in features])
    labels = [f["labels"] for f in features]

    # Pad labels to max length in batch with -100 (ignore index)
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return {"input_features": input_features, "labels": labels_padded}

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Setup training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
)

# Define metric computation
import jiwer

def compute_metrics(eval_pred):
    pred_ids, label_ids = eval_pred
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = jiwer.wer(label_str, pred_str)
    return {"wer": wer}

# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start fine-tuning
trainer.train()

# Save fine-tuned model and processor
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Fine-tuned model saved to {OUTPUT_DIR}")

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [20]:
!pip install torchcodec --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.8 MB/s eta 0:00:00


In [8]:
dataset.save_to_disk("/content/preprocessed_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/83 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/21 [00:00<?, ? examples/s]